In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# สร้าง Noise Models

<Accordion>
<AccordionItem title="เวอร์ชันของแพ็กเกจ">

โค้ดในหน้านี้พัฒนาขึ้นโดยใช้ข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
qiskit-aer~=0.17
```
</AccordionItem>
</Accordion>

  - To simulate noisy circuits, see [Exact and noisy simulation with Qiskit Aer primitives](/guides/simulate-with-qiskit-sdk-primitives).
หน้านี้แสดงวิธีใช้โมดูล [`noise`](https://qiskit.org/ecosystem/aer/apidocs/aer_noise.html) ของ Qiskit Aer เพื่อสร้าง noise model สำหรับการจำลอง quantum circuit ในสภาวะที่มีข้อผิดพลาด ซึ่งมีประโยชน์สำหรับการจำลอง quantum processor ที่มีสัญญาณรบกวน และสำหรับการศึกษาผลกระทบของสัญญาณรบกวนต่อการรันอัลกอริทึมควอนตัม

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator

# Import from Qiskit Aer noise module
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)

## โมดูล `noise` ของ Qiskit Aer

โมดูล `noise` ของ Qiskit Aer มีคลาส Python สำหรับสร้าง noise model แบบกำหนดเองสำหรับการจำลอง โดยมีสามคลาสหลักดังนี้

1. คลาส `NoiseModel` ซึ่งเก็บ noise model ที่ใช้สำหรับการจำลองแบบมีสัญญาณรบกวน
2. คลาส `QuantumError` ซึ่งอธิบาย CPTP gate error โดยสามารถนำไปใช้ได้:
    * หลังคำสั่ง *gate* หรือ *reset*
    * ก่อนคำสั่ง *measure*

3. คลาส `ReadoutError` ซึ่งอธิบาย classical readout error

## กำหนดค่า noise model จาก Backend

สามารถกำหนดค่า noise model โดยใช้พารามิเตอร์จากข้อมูลการสอบเทียบล่าสุดของ Backend จริงได้

> **Note:** ตัวอย่างเหล่านี้ใช้ mock backend `FakeSherbrooke` จาก `qiskit_ibm_runtime` แต่คุณสามารถลองใช้กับ backend จริงหรือ backend ปลอมใด ๆ ที่เข้ากันได้กับ Qiskit ก็ได้

In [2]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
noise_model = NoiseModel.from_backend(backend)

การทำเช่นนี้จะได้ noise model ที่ประมาณข้อผิดพลาดที่จะพบเมื่อใช้ Backend นั้น หากต้องการควบคุมพารามิเตอร์ของ noise model อย่างละเอียดกว่านี้ จำเป็นต้องสร้าง noise model เอง ตามที่อธิบายในส่วนที่เหลือของหน้านี้

## Quantum errors
แทนที่จะจัดการกับออบเจกต์ `QuantumError` โดยตรง มีฟังก์ชันช่วยหลายตัวสำหรับสร้าง quantum error ชนิดต่าง ๆ แบบพารามิเตอร์โดยอัตโนมัติ ฟังก์ชันเหล่านี้อยู่ในโมดูล `noise` และครอบคลุม error ประเภทที่ใช้บ่อยในงานวิจัยคอมพิวเตอร์ควอนตัม ชื่อฟังก์ชันและประเภท error ที่คืนค่ากลับมาได้แก่

| ฟังก์ชัน error มาตรฐาน | รายละเอียด |
| --- | --- |
| `kraus_error` | error channel CPTP แบบ n-qubit ทั่วไป ระบุเป็นรายการ Kraus matrices $[K_0, ...]$ |
| `mixed_unitary_error` | mixed unitary error แบบ n-qubit ระบุเป็นรายการ unitary matrices และความน่าจะเป็น $[(U_0, p_0),...]$ |
| `coherent_unitary_error` | coherent unitary error แบบ n-qubit ระบุเป็น unitary matrix เดี่ยว $U$ |
| `pauli_error` | Pauli error channel แบบ n-qubit (mixed unitary) ระบุเป็นรายการ Pauli และความน่าจะเป็น $[(P_0, p_0),...]$ |
| `depolarizing_error` | depolarizing error channel แบบ n-qubit กำหนดพารามิเตอร์ด้วยความน่าจะเป็น depolarization $p$ |
| `reset_error` | reset error แบบ single-qubit กำหนดพารามิเตอร์ด้วยความน่าจะเป็น $p_0, p_1$ ของการรีเซ็ตไปยังสถานะ $\vert0\rangle$, $\vert1\rangle$ |
| `thermal_relaxation_error` | thermal relaxation channel แบบ single qubit กำหนดพารามิเตอร์ด้วยค่าคงที่เวลาผ่อนคลาย $T_1$, $T_2$, เวลา gate $t$, และประชากรสถานะตื่นเต้นเชิงความร้อน $p_1$ |
| `phase_amplitude_damping_error` | generalized combined phase and amplitude damping error channel แบบ single-qubit ระบุด้วยพารามิเตอร์ amplitude damping $\lambda$, พารามิเตอร์ phase damping $\gamma$, และประชากรสถานะตื่นเต้นเชิงความร้อน $p_1$ |
| `amplitude_damping_error` | generalized amplitude damping error channel แบบ single-qubit ระบุด้วยพารามิเตอร์ amplitude damping $\lambda$, และประชากรสถานะตื่นเต้นเชิงความร้อน $p_1$ |
| `phase_damping_error` | phase damping error channel แบบ single-qubit ระบุด้วยพารามิเตอร์ phase damping $\gamma$ |

### รวม quantum errors
ออบเจกต์ `QuantumError` สามารถรวมกันได้โดยใช้ composition, tensor product, และ tensor expansion (tensor product ในลำดับกลับ) เพื่อสร้าง `QuantumError` ใหม่ ดังนี้

 * Composition: $\cal{E}(\rho)=\cal{E_2}(\cal{E_1}(\rho))$ เขียนเป็น `error = error1.compose(error2)`
 * Tensor product: $\cal{E}(\rho) =(\cal{E_1}\otimes\cal{E_2})(\rho)$ เขียนเป็น `error = error1.tensor(error2)`
 * Expand product: $\cal{E}(\rho) =(\cal{E_2}\otimes\cal{E_1})(\rho)$ เขียนเป็น `error = error1.expand(error2)`

### ตัวอย่าง
การสร้าง single-qubit bit-flip error 5%:

In [3]:
# Construct a 1-qubit bit-flip and phase-flip errors
p_error = 0.05
bit_flip = pauli_error([("X", p_error), ("I", 1 - p_error)])
phase_flip = pauli_error([("Z", p_error), ("I", 1 - p_error)])
print(bit_flip)
print(phase_flip)

QuantumError on 1 qubits. Noise circuits:
  P(0) = 0.05, Circuit = 
   ┌───┐
q: ┤ X ├
   └───┘
  P(1) = 0.95, Circuit = 
   ┌───┐
q: ┤ I ├
   └───┘
QuantumError on 1 qubits. Noise circuits:
  P(0) = 0.05, Circuit = 
   ┌───┐
q: ┤ Z ├
   └───┘
  P(1) = 0.95, Circuit = 
   ┌───┐
q: ┤ I ├
   └───┘


In [4]:
# Compose two bit-flip and phase-flip errors
bitphase_flip = bit_flip.compose(phase_flip)
print(bitphase_flip)

QuantumError on 1 qubits. Noise circuits:
  P(0) = 0.0025000000000000005, Circuit = 
   ┌───┐┌───┐
q: ┤ X ├┤ Z ├
   └───┘└───┘
  P(1) = 0.0475, Circuit = 
   ┌───┐┌───┐
q: ┤ X ├┤ I ├
   └───┘└───┘
  P(2) = 0.0475, Circuit = 
   ┌───┐┌───┐
q: ┤ I ├┤ Z ├
   └───┘└───┘
  P(3) = 0.9025, Circuit = 
   ┌───┐┌───┐
q: ┤ I ├┤ I ├
   └───┘└───┘


In [5]:
# Tensor product two bit-flip and phase-flip errors with
# bit-flip on qubit-0, phase-flip on qubit-1
error2 = phase_flip.tensor(bit_flip)
print(error2)

QuantumError on 2 qubits. Noise circuits:
  P(0) = 0.0025000000000000005, Circuit = 
     ┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ Z ├
     └───┘
  P(1) = 0.0475, Circuit = 
     ┌───┐
q_0: ┤ I ├
     ├───┤
q_1: ┤ Z ├
     └───┘
  P(2) = 0.0475, Circuit = 
     ┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ I ├
     └───┘
  P(3) = 0.9025, Circuit = 
     ┌───┐
q_0: ┤ I ├
     ├───┤
q_1: ┤ I ├
     └───┘


### Convert to and from QuantumChannel operators

We can also convert back and forth between `QuantumError` objects in Qiskit Aer and `QuantumChannel` objects in Qiskit.

In [6]:
# Convert to Kraus operator
bit_flip_kraus = Kraus(bit_flip)
print(bit_flip_kraus)

Kraus([[[-9.74679434e-01+0.j,  0.00000000e+00+0.j],
        [ 0.00000000e+00+0.j, -9.74679434e-01+0.j]],

       [[ 0.00000000e+00+0.j,  2.23606798e-01+0.j],
        [ 2.23606798e-01+0.j, -4.96506831e-17+0.j]]],
      input_dims=(2,), output_dims=(2,))


In [7]:
# Convert to Superoperator
phase_flip_sop = SuperOp(phase_flip)
print(phase_flip_sop)

SuperOp([[1. +0.j, 0. +0.j, 0. +0.j, 0. +0.j],
         [0. +0.j, 0.9+0.j, 0. +0.j, 0. +0.j],
         [0. +0.j, 0. +0.j, 0.9+0.j, 0. +0.j],
         [0. +0.j, 0. +0.j, 0. +0.j, 1. +0.j]],
        input_dims=(2,), output_dims=(2,))


In [8]:
# Convert back to a quantum error
print(QuantumError(bit_flip_kraus))

# Check conversion is equivalent to original error
QuantumError(bit_flip_kraus) == bit_flip

QuantumError on 1 qubits. Noise circuits:
  P(0) = 1.0, Circuit = 
   ┌───────┐
q: ┤ kraus ├
   └───────┘


True

### แปลงไปและกลับจาก QuantumChannel operators
สามารถแปลงระหว่างออบเจกต์ `QuantumError` ใน Qiskit Aer และออบเจกต์ `QuantumChannel` ใน Qiskit ได้ด้วย

In [9]:
# Measurement misassignment probabilities
p0given1 = 0.1
p1given0 = 0.05

ReadoutError([[1 - p1given0, p1given0], [p0given1, 1 - p0given1]])

ReadoutError([[0.95 0.05]
 [0.1  0.9 ]])

Readout errors may also be combined using `compose`, `tensor` and `expand`, like with quantum errors.

## Add errors to a noise model

When adding a quantum error to a noise model, we must specify the type of *instruction* that it acts on and what qubits to apply it to. There are two cases of quantum errors:

 1. All-qubit quantum error
 2. Specific qubit quantum error

### 1. All-qubit quantum error

This applies the same error to any occurrence of an instruction, regardless of which qubits it acts on.

It is added as `noise_model.add_all_qubit_quantum_error(error, instructions)`:

In [10]:
# Create an empty noise model
noise_model = NoiseModel()

# Add depolarizing error to all single qubit u1, u2, u3 gates
error = depolarizing_error(0.05, 1)
noise_model.add_all_qubit_quantum_error(error, ["u1", "u2", "u3"])

# Print noise model info
print(noise_model)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u1', 'u2', 'u3']
  Instructions with noise: ['u3', 'u1', 'u2']
  All-qubits errors: ['u1', 'u2', 'u3']


### 2. Specific qubit quantum error

This applies the error to any occurrence of an instruction acting on a specified list of qubits. Note that the order of the qubit matters: for example, an error applied to qubits [0, 1] for a two-qubit gate is different to one applied to qubits [1, 0].

It is added as `noise_model.add_quantum_error(error, instructions, qubits)`:

In [11]:
# Create an empty noise model
noise_model = NoiseModel()

# Add depolarizing error to all single qubit u1, u2, u3 gates on qubit 0 only
error = depolarizing_error(0.05, 1)
noise_model.add_quantum_error(error, ["u1", "u2", "u3"], [0])

# Print noise model info
print(noise_model)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u1', 'u2', 'u3']
  Instructions with noise: ['u3', 'u1', 'u2']
  Qubits with noise: [0]
  Specific qubit errors: [('u1', (0,)), ('u2', (0,)), ('u3', (0,))]


### Note on non-local qubit quantum error

`NoiseModel` does not support addition of non-local qubit quantum errors. They should be handled outside of `NoiseModel`. That suggests you should [write your own transpiler pass](/docs/guides/custom-transpiler-pass) (`TransformationPass`) and run the pass just before running the simulator if you need to insert your quantum errors into your circuit under your own conditions.

### Execute a noisy simulation with a noise model

The command `AerSimulator(noise_model=noise_model)` returns a simulator configured to the given noise model. In addition to setting the simulator's noise model, it also overrides the simulator's basis gates, according to the gates of the noise model.

## Noise model examples

We will now give some examples of noise models. For our demonstrations we use a simple test circuit generating an n-qubit GHZ state:

In [12]:
# System Specification
n_qubits = 4
circ = QuantumCircuit(n_qubits)

# Test Circuit
circ.h(0)
for qubit in range(n_qubits - 1):
    circ.cx(qubit, qubit + 1)
circ.measure_all()
print(circ)

        ┌───┐                ░ ┌─┐         
   q_0: ┤ H ├──■─────────────░─┤M├─────────
        └───┘┌─┴─┐           ░ └╥┘┌─┐      
   q_1: ─────┤ X ├──■────────░──╫─┤M├──────
             └───┘┌─┴─┐      ░  ║ └╥┘┌─┐   
   q_2: ──────────┤ X ├──■───░──╫──╫─┤M├───
                  └───┘┌─┴─┐ ░  ║  ║ └╥┘┌─┐
   q_3: ───────────────┤ X ├─░──╫──╫──╫─┤M├
                       └───┘ ░  ║  ║  ║ └╥┘
meas: 4/════════════════════════╩══╩══╩══╩═
                                0  1  2  3 


### Ideal simulation

In [13]:
# Ideal simulator and execution
sim_ideal = AerSimulator()
result_ideal = sim_ideal.run(circ).result()
plot_histogram(result_ideal.get_counts(0))

<Image src="../docs/images/guides/build-noise-models/extracted-outputs/bc713d11-755e-41e4-94f0-1ed76e3c2469-0.svg" alt="Output of the previous code cell" />

## Noise example 1: Basic bit-flip error noise model

Let's consider a simple toy noise model example common in quantum information theory research:

* When applying a single-qubit gate, flip the state of the qubit with probability `p_gate1`.
* When applying a two-qubit gate, apply single-qubit errors to each qubit.
* When resetting a qubit, reset to 1 instead of 0 with probability `p_reset`.
* When measuring a qubit, flip the state of the qubit with probability `p_meas`.

In [14]:
# Example error probabilities
p_reset = 0.03
p_meas = 0.1
p_gate1 = 0.05

# QuantumError objects
error_reset = pauli_error([("X", p_reset), ("I", 1 - p_reset)])
error_meas = pauli_error([("X", p_meas), ("I", 1 - p_meas)])
error_gate1 = pauli_error([("X", p_gate1), ("I", 1 - p_gate1)])
error_gate2 = error_gate1.tensor(error_gate1)

# Add errors to noise model
noise_bit_flip = NoiseModel()
noise_bit_flip.add_all_qubit_quantum_error(error_reset, "reset")
noise_bit_flip.add_all_qubit_quantum_error(error_meas, "measure")
noise_bit_flip.add_all_qubit_quantum_error(error_gate1, ["u1", "u2", "u3"])
noise_bit_flip.add_all_qubit_quantum_error(error_gate2, ["cx"])

print(noise_bit_flip)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u1', 'u2', 'u3']
  Instructions with noise: ['measure', 'cx', 'reset', 'u2', 'u3', 'u1']
  All-qubits errors: ['reset', 'measure', 'u1', 'u2', 'u3', 'cx']


### Execute the noisy simulation

In [15]:
# Create noisy simulator backend
sim_noise = AerSimulator(noise_model=noise_bit_flip)

# Transpile circuit for noisy basis gates
passmanager = generate_preset_pass_manager(
    optimization_level=3, backend=sim_noise
)
circ_tnoise = passmanager.run(circ)

# Run and get counts
result_bit_flip = sim_noise.run(circ_tnoise).result()
counts_bit_flip = result_bit_flip.get_counts(0)

# Plot noisy output
plot_histogram(counts_bit_flip)

<Image src="../docs/images/guides/build-noise-models/extracted-outputs/abeb9f09-d762-406d-983e-0357ade59636-0.svg" alt="Output of the previous code cell" />

### Readout error
Classical readout error ระบุเป็นรายการ assignment probability vectors $P(A|B)$:

  * $A$ คือค่า classical bit ที่ *บันทึกไว้*
  * $B$ คือค่า bit *จริง* ที่ได้จากการวัด

ตัวอย่างเช่น สำหรับหนึ่ง Qubit: $ P(A|B) = [P(A|0), P(A|1)]$

In [16]:
# T1 and T2 values for qubits 0-3
T1s = np.random.normal(
    50e3, 10e3, 4
)  # Sampled from normal distribution mean 50 microsec
T2s = np.random.normal(
    70e3, 10e3, 4
)  # Sampled from normal distribution mean 50 microsec

# Truncate random T2s <= T1s
T2s = np.array([min(T2s[j], 2 * T1s[j]) for j in range(4)])

# Instruction times (in nanoseconds)
time_u1 = 0  # virtual gate
time_u2 = 50  # (single X90 pulse)
time_u3 = 100  # (two X90 pulses)
time_cx = 300
time_reset = 1000  # 1 microsecond
time_measure = 1000  # 1 microsecond

# QuantumError objects
errors_reset = [
    thermal_relaxation_error(t1, t2, time_reset) for t1, t2 in zip(T1s, T2s)
]
errors_measure = [
    thermal_relaxation_error(t1, t2, time_measure) for t1, t2 in zip(T1s, T2s)
]
errors_u1 = [
    thermal_relaxation_error(t1, t2, time_u1) for t1, t2 in zip(T1s, T2s)
]
errors_u2 = [
    thermal_relaxation_error(t1, t2, time_u2) for t1, t2 in zip(T1s, T2s)
]
errors_u3 = [
    thermal_relaxation_error(t1, t2, time_u3) for t1, t2 in zip(T1s, T2s)
]
errors_cx = [
    [
        thermal_relaxation_error(t1a, t2a, time_cx).expand(
            thermal_relaxation_error(t1b, t2b, time_cx)
        )
        for t1a, t2a in zip(T1s, T2s)
    ]
    for t1b, t2b in zip(T1s, T2s)
]

# Add errors to noise model
noise_thermal = NoiseModel()
for j in range(4):
    noise_thermal.add_quantum_error(errors_reset[j], "reset", [j])
    noise_thermal.add_quantum_error(errors_measure[j], "measure", [j])
    noise_thermal.add_quantum_error(errors_u1[j], "u1", [j])
    noise_thermal.add_quantum_error(errors_u2[j], "u2", [j])
    noise_thermal.add_quantum_error(errors_u3[j], "u3", [j])
    for k in range(4):
        noise_thermal.add_quantum_error(errors_cx[j][k], "cx", [j, k])

print(noise_thermal)

NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx', 'u2', 'u3']
  Instructions with noise: ['cx', 'measure', 'reset', 'u2', 'u3']
  Qubits with noise: [0, 1, 2, 3]
  Specific qubit errors: [('reset', (0,)), ('reset', (1,)), ('reset', (2,)), ('reset', (3,)), ('measure', (0,)), ('measure', (1,)), ('measure', (2,)), ('measure', (3,)), ('u2', (0,)), ('u2', (1,)), ('u2', (2,)), ('u2', (3,)), ('u3', (0,)), ('u3', (1,)), ('u3', (2,)), ('u3', (3,)), ('cx', (0, 0)), ('cx', (0, 1)), ('cx', (0, 2)), ('cx', (0, 3)), ('cx', (1, 0)), ('cx', (1, 1)), ('cx', (1, 2)), ('cx', (1, 3)), ('cx', (2, 0)), ('cx', (2, 1)), ('cx', (2, 2)), ('cx', (2, 3)), ('cx', (3, 0)), ('cx', (3, 1)), ('cx', (3, 2)), ('cx', (3, 3))]


### Execute the noisy simulation

In [17]:
# Run the noisy simulation
sim_thermal = AerSimulator(noise_model=noise_thermal)

# Transpile circuit for noisy basis gates
passmanager = generate_preset_pass_manager(
    optimization_level=3, backend=sim_thermal
)
circ_tthermal = passmanager.run(circ)

# Run and get counts
result_thermal = sim_thermal.run(circ_tthermal).result()
counts_thermal = result_thermal.get_counts(0)

# Plot noisy output
plot_histogram(counts_thermal)

<Image src="../docs/images/guides/build-noise-models/extracted-outputs/ff52bf52-1323-40fc-a631-2b1889b21b20-0.svg" alt="Output of the previous code cell" />

Readout error ยังสามารถรวมกันได้โดยใช้ `compose`, `tensor` และ `expand` เช่นเดียวกับ quantum error

## การเพิ่ม error เข้าไปใน noise model
เมื่อเพิ่ม quantum error เข้าไปใน noise model ต้องระบุประเภท *instruction* ที่มันทำงานและ Qubit ที่จะนำไปใช้ มี quantum error สองกรณีดังนี้

 1. All-qubit quantum error
 2. Specific qubit quantum error

### 1. All-qubit quantum error
ใช้ error เดียวกันกับทุก instruction ที่เกิดขึ้น ไม่ว่า Qubit ใดจะทำงานอยู่

เพิ่มด้วย `noise_model.add_all_qubit_quantum_error(error, instructions)`: